In [ ]:
# EC-GBM Implementation in Python

import numpy as np
import matplotlib.pyplot as plt

# EC-GBM Configuration
T = 100  # Number of time steps
S0 = 1000  # Initial stock price
mu = 0.02  # Drift
sigma = 0.05  # Volatility
M = 10000000  # Number of Monte Carlo trajectories
threshold = 0.01  # Entropy reduction threshold

# Function to generate a GBM trajectory
def generate_gbm_trajectory(S0, mu, sigma, T):
    dt = 1 / T
    W = np.random.normal(0, np.sqrt(dt), T).cumsum()
    time = np.linspace(0, 1, T)
    S = S0 * np.exp((mu - 0.5 * sigma**2) * time + sigma * W)
    return S

# Compute Shannon entropy for a given set of trajectories
def compute_entropy(data, bins=20):
    hist, _ = np.histogram(data, bins=bins, density=True)
    probs = hist / hist.sum()
    entropy = -np.sum(probs * np.log(probs + 1e-10))  # Avoid log(0)
    return entropy

# Main EC-GBM algorithm
def ec_gbm(S0, mu, sigma, T, M, threshold):
    initial_trajectory = generate_gbm_trajectory(S0, mu, sigma, T)
    reference_trajectory = initial_trajectory.copy()
    selected_paths = []

    for _ in range(M):
        new_trajectory = generate_gbm_trajectory(S0, mu, sigma, T)
        combined_trajectory = np.concatenate((reference_trajectory, new_trajectory))
        
        original_entropy = compute_entropy(reference_trajectory)
        new_entropy = compute_entropy(combined_trajectory)
        
        if original_entropy - new_entropy > threshold:
            selected_paths.append(new_trajectory)
            reference_trajectory = combined_trajectory

    selected_paths = np.array(selected_paths)
    if selected_paths.size > 0:
        mean_prediction = selected_paths.mean(axis=0)
    else:
        mean_prediction = initial_trajectory

    return initial_trajectory, mean_prediction

# Run EC-GBM
initial, ec_gbm_prediction = ec_gbm(S0, mu, sigma, T, M, threshold)

# Plotting results
plt.figure(figsize=(10, 6))
time = np.linspace(0, 1, T)
plt.plot(time, initial, label="Reference Series", color="blue")
plt.plot(time, ec_gbm_prediction, label="EC-GBM Prediction", linestyle="--", color="green")
plt.title("EC-GBM Stock Price Prediction")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid()
plt.show()

